In [1]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# reproducibility
random.seed(42)
torch.manual_seed(42)

with open("names.txt", "r") as f:
    names = [line.strip().lower() for line in f if line.strip()]

print("number of names:", len(names))
print("first 10:", names[:10])
print("shortest length:", min(len(name) for name in names))
print("longest length:", max(len(name) for name in names))

number of names: 32033
first 10: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']
shortest length: 2
longest length: 15


In [2]:
chars = ".abcdefghijklmnopqrstuvwxyz"

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(chars)
MAX_LENGTH = max(len(name) for name in names)

print("vocab size:", VOCAB_SIZE)
print("max context length:", MAX_LENGTH)
print(stoi)

vocab size: 27
max context length: 15
{'.': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26}


In [3]:
random.shuffle(names)

split = int(len(names) * 0.9)

train_names = names[:split]
val_names = names[split:]

print("train names:", len(train_names))
print("validation names:", len(val_names))

train names: 28829
validation names: 3204


In [4]:
def build_dataset(names):
    X = []
    Y = []

    for name in names:
        # 一开始什么都没有
        context = [stoi["."]] * MAX_LENGTH

        # name 后面加 "."，表示 STOP
        for ch in name + ".":
            X.append(context.copy())
            Y.append(stoi[ch])

            # 把刚刚看到的字符加入 context
            context = context[1:] + [stoi[ch]]

    X = torch.tensor(X, dtype=torch.long)
    Y = torch.tensor(Y, dtype=torch.long)

    return X, Y


X_train, Y_train = build_dataset(train_names)
X_val, Y_val = build_dataset(val_names)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

X_train: torch.Size([205280, 15])
Y_train: torch.Size([205280])
X_val: torch.Size([22866, 15])
Y_val: torch.Size([22866])


In [11]:
print(X_train[2])
print(X_train[3])
print(X_train[4])
print(X_train[5])

tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 25, 21])
tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 25, 21,  8])
tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 25, 21,  8,  5])
tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 25, 21,  8,  5, 14])


In [12]:
BATCH_SIZE = 512

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, Y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)

device: mps


In [13]:
class NameMLP(nn.Module):
    def __init__(self, max_length, vocab_size, hidden_size=256):
        super().__init__()

        input_size = max_length * vocab_size

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x shape:
        # [batch_size, MAX_LENGTH]

        # one-hot:
        # [batch_size, MAX_LENGTH, VOCAB_SIZE]
        x = F.one_hot(x, num_classes=VOCAB_SIZE).float()

        # flatten:
        # [batch_size, MAX_LENGTH * VOCAB_SIZE]
        x = x.view(x.shape[0], -1)

        x = self.fc1(x)
        x = F.relu(x)
        logits = self.fc2(x)

        return logits


model = NameMLP(
    max_length=MAX_LENGTH,
    vocab_size=VOCAB_SIZE,
    hidden_size=256
).to(device)

print(model)

num_params = sum(p.numel() for p in model.parameters())
print("number of parameters:", f"{num_params:,}")

NameMLP(
  (fc1): Linear(in_features=405, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=27, bias=True)
)
number of parameters: 110,875


## What does a Linear do? The right mental model

In [18]:
fc = nn.Linear(3, 2)
x = torch.tensor([1.0, 2.0, 3.0])

y = fc(x)

print(y)
print(fc.weight)
print(fc.bias)

tensor([-2.3371,  1.8121], grad_fn=<ViewBackward0>)
Parameter containing:
tensor([[-0.0564, -0.2278, -0.4356],
        [ 0.0180,  0.3769,  0.2673]], requires_grad=True)
Parameter containing:
tensor([-0.5184,  0.2386], requires_grad=True)


### temperature牛逼的实现。
### softmax（dim）来决定哪个dim来算数

In [21]:
def generate_name(model, temperature=1.0, max_length=MAX_LENGTH):
    model.eval()

    context = [stoi["."]] * MAX_LENGTH
    generated = []

    with torch.no_grad():
        for _ in range(max_length):

            x = torch.tensor(
                [context],
                dtype=torch.long,
                device=device
            )

            logits = model(x)[0]

            logits = logits / temperature
            probs = F.softmax(logits, dim=0)

            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_char = itos[next_idx]

            if next_char == ".":
                break

            generated.append(next_char)

            context = context[1:] + [next_idx]

    return "".join(generated)


for _ in range(20):
    print(generate_name(model))

lfic
yrwgcwyhxyubzis
tqluhjocxjcenhd
aohrvngmvnmjcqg
yakag
svsbdwqwtzveucv
bkapdjjztrvpnaa
sdupmp
oykepkcdtbjoijf
qmsa
esdcomcdymqkrel
jtkkenc
hhnlsyxtdj
bcyavvjyqgopabi

atqfhddfsjqjtbl
pshblabf
giiutwuxoacqjin
ouaqisfyrr
idbaaqgwlbaucuw


### loss感觉刚开始也不大，然后最后开始平了？关于loss，是越接近0越好吗？有什么理论？大家怎么知道理论最低loss？

1. 随机瞎猜 27 个字符，如果每个都是：(1/27), loss是多少？ 3.3
2. 真实世界是
B: 0.5
C: 0.5

loss最低值是0吗？
不是！
是 -log(0.5)=0.693

## this is 理论最低值！baby！！
## “有限训练集的 entropy” ≠ “真实语言的 entropy”！！ 因为人类文本只是“真实”的一个子集！
换句话说，我们的名字example能够足够逼近“真实”的名字概率分布。
但是人类产生的文本，只是“真实”的一个非常小的子集，因为复杂度爆炸了。

我的意思是，假如有一个只是关于名字的“世界”，因为这个问题的space很小，我们的name.txt足够逼真了，跟这个“真实世界”足够接近了，对吗？

In [33]:
from collections import defaultdict, Counter
import math

counts = defaultdict(Counter)

for x, y in zip(X_train.tolist(), Y_train.tolist()):
    counts[tuple(x)][y] += 1

total_loss = 0
total_examples = 0

for context, target_counts in counts.items():
    n = sum(target_counts.values())

    for target, count in target_counts.items():
        p = count / n
        total_loss += count * (-math.log(p))
        total_examples += count

theoretical_train_loss = total_loss / total_examples

print("empirical theoretical minimum:", theoretical_train_loss)

empirical theoretical minimum: 1.4281315646563795


In [22]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10

for epoch in range(EPOCHS):

    # ---------- training ----------
    model.train()

    total_train_loss = 0
    total_train_examples = 0

    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        logits = model(X_batch)

        loss = F.cross_entropy(l
                               ogits, Y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * X_batch.shape[0]
        total_train_examples += X_batch.shape[0]

    train_loss = total_train_loss / total_train_examples

    # ---------- validation ----------
    model.eval()

    total_val_loss = 0
    total_val_examples = 0

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            logits = model(X_batch)
            loss = F.cross_entropy(logits, Y_batch)

            total_val_loss += loss.item() * X_batch.shape[0]
            total_val_examples += X_batch.shape[0]

    val_loss = total_val_loss / total_val_examples

    print(
        f"epoch {epoch + 1:2d} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_loss:.4f}"
    )

epoch  1 | train loss 2.3942 | val loss 2.2305
epoch  2 | train loss 2.1951 | val loss 2.1706
epoch  3 | train loss 2.1441 | val loss 2.1332
epoch  4 | train loss 2.1077 | val loss 2.1060
epoch  5 | train loss 2.0812 | val loss 2.0876
epoch  6 | train loss 2.0589 | val loss 2.0747
epoch  7 | train loss 2.0414 | val loss 2.0630
epoch  8 | train loss 2.0264 | val loss 2.0513
epoch  9 | train loss 2.0124 | val loss 2.0438
epoch 10 | train loss 2.0002 | val loss 2.0370


In [23]:
for _ in range(30):
    print(generate_name(model, temperature=1.0))

kiyo
mariela
paireon
armckelle
aroma
arisha
sadefi
naxxson
yous
caiden
azabrah
niya
zhad
taalee
hodinnis
droptne
kamphton
reget
alaton
satalaina
angely
raderi
brinel
crotella
adiami
konzi
kanalia
reckonn
shaser
yrand


In [24]:
train_name_set = set(train_names)

for _ in range(30):
    name = generate_name(model, temperature=1.0)

    if name in train_name_set:
        status = "SEEN"
    else:
        status = "NEW"

    print(f"{name:20s} {status}")

korri                SEEN
joobi                NEW
koyven               NEW
kayvi                NEW
rosi                 NEW
cagyrelli            NEW
aluina               NEW
margana              NEW
yayne                NEW
sola                 SEEN
mahnabi              NEW
rami                 NEW
christia             NEW
andian               NEW
dan                  SEEN
den                  NEW
imila                NEW
tamghon              NEW
inem                 NEW
zathyah              NEW
demman               NEW
hamberron            NEW
prcterys             NEW
jamana               NEW
sanah                SEEN
sham                 SEEN
ccalpir              NEW
jolei                SEEN
zavira               NEW
kynna                NEW
